## Julieta Madrigal Flores - 744029
## Lunes 4 de mayo del 2026

In [1]:
import pandas as pd
import numpy as np

# **BOOTSTRAPPING**

### **EDA - tomar el dataset**

In [2]:
data = pd.read_excel('Motor Trend Car Road Tests.xlsx')
data.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [3]:
data.describe()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
count,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.0000
mean,20.090625,6.187500,230.721875,146.687500,3.596563,3.217250,17.848750,0.437500,0.406250,3.687500,2.8125
std,6.026948,1.785922,123.938694,68.562868,0.534679,0.978457,1.786943,0.504016,0.498991,0.737804,1.6152
min,10.400000,4.000000,71.100000,52.000000,2.760000,1.513000,14.500000,0.000000,0.000000,3.000000,1.0000
25%,15.425000,4.000000,120.825000,96.500000,3.080000,2.581250,16.892500,0.000000,0.000000,3.000000,2.0000
50%,19.200000,6.000000,196.300000,123.000000,3.695000,3.325000,17.710000,0.000000,0.000000,4.000000,2.0000
75%,22.800000,8.000000,326.000000,180.000000,3.920000,3.610000,18.900000,1.000000,1.000000,4.000000,4.0000
max,33.900000,8.000000,472.000000,335.000000,4.930000,5.424000,22.900000,1.000000,1.000000,5.000000,8.0000


### **Definir X y Y**

In [4]:
y = data['mpg']
X = data[['hp','qsec']]

### **Realizar regresión**

In [5]:
import statsmodels.api as sm

X_const = sm.add_constant(X)  # agrega el intercepto (β₀)
modelo = sm.OLS(y, X_const).fit()

In [6]:
# Ver resumen completo con IC al 95%
modelo.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        22:42:14   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.000      25.615      71.033
hp            -0.0846      0.014     -6.071      0.000      -0.113      -0.056
qsec          -0.8866      0.535     -1.658      0.108      -1.980       0.207
==============================================================================
Omnibus:                        5.102   Durbin-Watson:                   1.355
Prob(Omnibus):                  0.078   Jarque-Bera (JB):                3.987
Skew:                           0.858   Prob(JB):                        0.136
Kurtosis:                       3.220   Cond. No.                     2.72e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.72e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### **Calcular intervalos de confianza de los betas**

In [7]:
# Solo los intervalos de confianza = 95% de confianza
# [límite inferior = 0 , límite superior = 1]
modelo.conf_int(alpha=0.05)

,0,1
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


### **Crear 1000 muestras de bootstrap con m observaciones con reemplazo y  realizar la regresión OTRA VEZ**

In [8]:
from sklearn.linear_model import LinearRegression

In [9]:
np.random.seed(42)
n_bootstrap = 1000 # número de iteraciones = muestras bootstrap que se van a crear
n = len(data) # Cada muestra bootstrap debe de tener el mismo tamaño que el original

# lista vacía donde vas a ir guardando los betas de cada iteración
betas_boot = []

for i in range(n_bootstrap):
    # Toma una muestra aleatoria con reemplazo del dataset original
    # Con reemplazo = un mismo carro puede aparecer varias veces en la muestra y otros pueden no aparecer porque tomas la muestra, la ves y dices "va, ya la vi" y la regresas
    # Sigue siendo de 32 filas pero distinta cada vez
    muestra = data.sample(n=n, replace=True)

    X_boot = muestra[['hp', 'qsec']]
    y_boot = muestra['mpg']

    # Hace la regresion con las matrices actualizadas
    # Entrena una regresión lineal con esa muestra
    # Arroja betas distintos cada vez porque los datos cambian con cada iteración
    lr_boot = LinearRegression().fit(X_boot, y_boot)

    # Agrega las nuevas betas
    # Guarda los 3 betas de esa iteración en la lista: [b0, b1, b2] =  1000 filas con 3 columnas
    betas_boot.append([lr_boot.intercept_,lr_boot.coef_[0],lr_boot.coef_[1]])

# Convierte la lista en un arreglo de numpy de (1000 × 3)
betas_boot = np.array(betas_boot)

In [10]:
betas_boot

array([[ 5.94478918e+01, -9.76086636e-02, -1.44712695e+00],
       [ 5.64310306e+01, -9.54026798e-02, -1.23409336e+00],
       [ 6.57094280e+01, -1.18781752e-01, -1.54608740e+00],
       ...,
       [ 5.71490975e+01, -1.07407397e-01, -1.21491970e+00],
       [ 4.92240122e+01, -8.86184630e-02, -9.54707733e-01],
       [ 4.12373113e+01, -6.20832138e-02, -7.01686212e-01]])

In [11]:
betas_boot.shape

(1000, 3)

### **Calcular el promedio de cada beta**

In [12]:
[promedio_b0, promedio_b1, promedio_b2] = np.mean(betas_boot, axis=0)
promedio_b0, promedio_b1, promedio_b2

(np.float64(50.20333012092137),
 np.float64(-0.08813593213276785),
 np.float64(-0.9691262403127736))

### **Calcular el promedio de sus desviaciones estándar**

In [13]:
[desv_b0, desv_b1, desv_b2] = np.std(betas_boot, axis=0)
desv_b0, desv_b1, desv_b2

(np.float64(11.130468570891562),
 np.float64(0.0167785651199201),
 np.float64(0.5329147856736497))

### **Abrir un intervalo de confianza**

In [14]:
int_conf_b0_lower= promedio_b0 - 2*desv_b0
int_conf_b0_upper= promedio_b0 + 2*desv_b0

int_conf_b0_lower, int_conf_b0_upper

(np.float64(27.942392979138248), np.float64(72.4642672627045))

In [15]:
int_conf_b1_lower= promedio_b1 - 2*desv_b1
int_conf_b1_upper= promedio_b1 + 2*desv_b1

int_conf_b1_lower, int_conf_b1_upper

(np.float64(-0.12169306237260805), np.float64(-0.05457880189292765))

In [16]:
int_conf_b2_lower= promedio_b2 - 2*desv_b2
int_conf_b2_upper= promedio_b2 + 2*desv_b2

int_conf_b2_lower, int_conf_b2_upper

(np.float64(-2.034955811660073), np.float64(0.09670333103452577))

### **Comparar el primer intervalo con el intervalo con bootstrapping**

Al comparar ambos métodos, los resultados son casi iguales entre sí.


Para **b0 (intercepto)**, el método sin boostrapping arrojó un intervalo de [25.61, 71.03],mientras que el bootstrap dio [27.94, 72.46] — prácticamente el mismo rango.


Para **b1 (hp)**, el simple fue [-0.1131, -0.0561] y el bootstrap [-0.1217, -0.0546],ambos con negativos y de magnitudes similares. Además, los dos intervalos son negativos, lo que significa que a mayor potencia (hp), el carro consume más gasolina y baja su mpg.


Para **b2 (qsec)**, el simple fue [-1.98, 0.21] y el bootstrap [-2.03, 0.10],de nuevo muy cercanos.

El hecho de que ambos resultados sean tan similares indica que los supuestos del modelo clásico (normalidad de los errores, homocedasticidad) se cumplen razonablemente bien en este dataset. Es decir, el método simple  (statsmodels) asume que los errores siguen una distribución normal; mientras que el bootstrap no asume nada, solo remuestrea y como ambos dieron resultados similares, significa que esa suposición de normalidad sí se cumple en tus datos.


# **Aggregating**

In [17]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
data = pd.read_excel('Motor Trend Car Road Tests.xlsx')
data.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [19]:
# Preparar datos - eliminar columna 'model'
data_clean = data.drop(columns=['model'])

In [20]:
# Definir X (todas las columnas excepto mpg) e y
y = data_clean['mpg']
X = data_clean.drop(columns=['mpg'])

# Ver las columnas disponibles para seleccionar al azar
available_features = list(X.columns)

available_features

['cyl', 'disp', 'hp', 'drat', 'wt', 'qsec', 'vs', 'am', 'gear', 'carb']

In [21]:
print(f"Total: {len(available_features)} columnas")

Total: 10 columnas


In [22]:
# Train/test split 50%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [23]:
n_models = 1000 # mil modelos
n_features = 3  # 3 columnas al azar por modelo

# Listas para guardar resultados
models = []  # guardar cada modelo entrenado
feature_sets = []  # guardar qué columnas usó cada modelo
r2_train_scores = []  # guardar R2 de cada modelo en TRAIN
r2_test_scores = []  # guardar R2 de cada modelo en TEST

# Crear 1000 modelos con 3 columnas aleatorias (sin reemplazo)
np.random.seed(42)

In [24]:
all_test_predictions = []  # LISTA NUEVA para guardar predicciones de cada modelo

for i in range(n_models):
    selected_features = np.random.choice(available_features, size=n_features, replace=False)
    feature_sets.append(selected_features)

    X_train_subset = X_train[selected_features]
    X_test_subset = X_test[selected_features]

     # Entrenar modelo
    model = LinearRegression()
    model.fit(X_train_subset, y_train)
    models.append(model)

    y_train_pred = model.predict(X_train_subset)
    r2_train = r2_score(y_train, y_train_pred)
    r2_train_scores.append(r2_train)

    y_test_pred = model.predict(X_test_subset)  # esto es un array de tamaño (n_test,)
    r2_test = r2_score(y_test, y_test_pred)
    r2_test_scores.append(r2_test)

    all_test_predictions.append(y_test_pred)  # GUARDAR las predicciones

# Convertir a numpy array
all_test_predictions = np.array(all_test_predictions)  # forma: (1000, n_test)

In [25]:
# Mostrar resultados
print(f"Promedio R² en TRAIN de los 1000 modelos: {np.mean(r2_train_scores):.4f}")
print(f"Promedio R² en TEST de los 1000 modelos: {np.mean(r2_test_scores):.4f}")

Promedio R² en TRAIN de los 1000 modelos: 0.7853
Promedio R² en TEST de los 1000 modelos: 0.6222


In [26]:
# Promedio de cada fila (promedio de las predicciones de cada modelo a través de todas las observaciones)
promedio_por_fila = np.mean(all_test_predictions, axis=1)
print(promedio_por_fila)

[18.72669966 18.39172028 17.72128899 18.4750095  17.23669391 17.62670937
 17.10140262 17.73695874 16.94450955 16.95901024 17.73670589 18.25474169
 17.72210185 17.08896926 18.32711983 18.58857677 17.33094837 18.72669966
 17.1288626  18.42702985 18.12141616 17.9772709  17.45715012 17.19060665
 17.08185551 16.9286708  17.74499424 17.71245018 16.96585873 18.11884039
 17.08185551 17.40224653 18.69850277 17.42994416 16.96585873 17.09125494
 18.25474169 17.16264462 18.3464346  18.72669966 17.54521928 18.67965847
 18.25474169 17.12436119 17.85180027 16.9286708  18.65093621 17.73695874
 17.33414765 17.33094837 18.29232943 17.91907258 17.40224653 18.14016679
 16.95901024 17.68503875 18.27672094 17.32117106 16.99070617 17.13044863
 16.44359334 17.13200508 17.61512424 17.39051109 17.45715012 18.25474169
 17.23669391 18.14016679 18.69850277 17.9772709  16.96585873 18.72669966
 17.54878457 18.3464346  18.38122398 17.91907258 18.38122398 17.12436119
 17.71245018 18.14934765 17.60472959 18.49194576 17

In [27]:
promedio_por_fila = np.array(promedio_por_fila) # te arroja mil promedios
promedio = promedio_por_fila.mean() #


In [28]:
aggregated_predictions = np.mean(all_test_predictions, axis=0)
r2_promedio = r2_score(y_test, aggregated_predictions)
r2_promedio

0.7816874948553102

# **RANDOM FOREST**

In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

In [63]:
from sklearn.model_selection import GridSearchCV, KFold
# Definir hiperparámetros
param_grid = {
    'n_estimators': [10, 50, 100, 200, 300],      # cantidad de árboles
    'max_leaf_nodes': [None, 10, 20, 30, 50],      # cantidad de hojas
    'max_depth': [None, 3, 5, 10, 15]              # profundidad
}

In [64]:
# K-Fold k=10 y GridSearchCV
kf = KFold(n_splits=10, shuffle=True, random_state=42)

rf = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1 # va a tener más costo computacional pero lo va a hacer más rápido
)

grid_search.fit(X, y)

GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 3, 5, 10, 15],
                         'max_leaf_nodes': [None, 10, 20, 30, 50],
                         'n_estimators': [10, 50, 100, 200, 300]},
             scoring='r2')

In [65]:
# Resultados
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"R² promedio por cada K-Fold k=10: {grid_search.best_score_:.4f}")

Mejores hiperparámetros: {'max_depth': 3, 'max_leaf_nodes': 10, 'n_estimators': 300}
R² promedio por cada K-Fold k=10: 0.4999


In [66]:
# Usar el dataset completo
y = data_clean['mpg']
X = data_clean.drop(columns=['mpg'])

print(X.shape)  # debe ser (32, 10)

(32, 10)


In [67]:
mejor_modelo = grid_search.best_estimator_ # extrae el modelo que ya entrenó GridSearchCV con los mejores hiperparámetros encontrados
y_pred = mejor_modelo.predict(X)
r2_final = r2_score(y, y_pred) # compara las predicciones contra los valores reales de mpg
print(f"R² del mejor modelo: {r2_final:.4f}")

R² del mejor modelo: 0.9646


# BOOSTING

In [47]:
from sklearn.ensemble import GradientBoostingRegressor

In [59]:
# Definir hiperparámetros
param_grid = {
    'n_estimators': [10, 50, 100, 200,300],      # cantidad de árboles
    'max_leaf_nodes': [None, 10, 20, 30, 50,75],      # cantidad de hojas
    'max_depth': [None, 3, 5, 10, 15]              # profundidad
}

In [60]:
# K-Fold k=10 y GridSearchCV
kf = KFold(n_splits=10, shuffle=True, random_state=42)

gb = GradientBoostingRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=kf,
    scoring='r2',
    n_jobs=-1 # va a tener más costo computacional pero lo va a hacer más rápido
)

grid_search.fit(X, y)

GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=GradientBoostingRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 3, 5, 10, 15],
                         'max_leaf_nodes': [None, 10, 20, 30, 50, 75],
                         'n_estimators': [10, 50, 100, 200, 300]},
             scoring='r2')

In [61]:
# Resultados
print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"R² promedio por cada K-Fold k=10: {grid_search.best_score_:.4f}")

Mejores hiperparámetros: {'max_depth': 3, 'max_leaf_nodes': None, 'n_estimators': 50}
R² promedio por cada K-Fold k=10: 0.5873


In [62]:
mejor_modelo = grid_search.best_estimator_ # extrae el modelo que ya entrenó GridSearchCV con los mejores hiperparámetros encontrados
y_pred = mejor_modelo.predict(X)
r2_final = r2_score(y, y_pred) # compara las predicciones contra los valores reales de mpg
print(f"R² del mejor modelo: {r2_final:.4f}")

R² del mejor modelo: 0.9982
